In [10]:
import pandas as pd
import openpyxl
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

## SocioEcon 2050 Summary by Scenario

In [ ]:
base = "../data/processed_data"
out_path = "../SocioEcon_2050_Summary_ByScenario.xlsx"

sum_vars = [
    'total_residential_units', 'total_occ_units',
    'occ_units_low_inc', 'occ_units_med_inc', 'occ_units_high_inc',
    'total_persons',
    'emp_retail', 'emp_srvc', 'emp_rec', 'emp_game', 'emp_other'
]

# Load and sum each scenario
sums = {}
for alt in [1, 2, 3, 4]:
    df = pd.read_csv(f"{base}/Alternative_{alt}_2050/SocioEcon_Summer.csv")
    sums[alt] = {v: int(df[v].sum()) for v in sum_vars}

# Build main data rows
rows = []
for v in sum_vars:
    rows.append({
        'variable': v,
        'scenario_1_sum': sums[1][v],
        'scenario_2_sum': sums[2][v],
        'scenario_3_sum': sums[3][v],
        'scenario_4_sum': sums[4][v],
        'diff_2_vs_1': sums[2][v] - sums[1][v],
        'diff_3_vs_1': sums[3][v] - sums[1][v],
        'diff_4_vs_1': sums[4][v] - sums[1][v],
    })

# Build income % rows
inc_vars = ['occ_units_low_inc', 'occ_units_med_inc', 'occ_units_high_inc']
inc_labels = ['% Low Income', '% Middle Income', '% High Income']
inc_rows = []
for label, v in zip(inc_labels, inc_vars):
    row = {'income_category': label}
    for alt in [1, 2, 3, 4]:
        total_inc = sum(sums[alt][iv] for iv in inc_vars)
        row[f'scenario_{alt}_pct'] = sums[alt][v] / total_inc if total_inc else 0
    inc_rows.append(row)

wb = openpyxl.Workbook()
bold = Font(bold=True)

# ── Sheet 1: Summary ──────────────────────────────────────────────────────────
ws1 = wb.active
ws1.title = 'SocioEcon_2050_Summary'

ws1.append([None, None, None, None, None, 'Absolute Difference', None, None, '% Difference', None, None])
ws1.append(['variable', 'scenario_1_sum', 'scenario_2_sum', 'scenario_3_sum', 'scenario_4_sum',
            'diff_2_vs_1', 'diff_3_vs_1', 'diff_4_vs_1', 2, 3, 4])

for i, row in enumerate(rows, start=3):
    ws1.append([
        row['variable'],
        row['scenario_1_sum'], row['scenario_2_sum'],
        row['scenario_3_sum'], row['scenario_4_sum'],
        row['diff_2_vs_1'], row['diff_3_vs_1'], row['diff_4_vs_1'],
        f'=F{i}/$B{i}', f'=G{i}/$B{i}', f'=H{i}/$B{i}',
    ])

# Format % difference columns
for row in ws1.iter_rows(min_row=3, max_row=2 + len(rows), min_col=9, max_col=11):
    for cell in row:
        cell.number_format = '0.0%'

# Bold headers, merge span headers
for cell in ws1[1]: cell.font = bold
for cell in ws1[2]: cell.font = bold
ws1.merge_cells('F1:H1')
ws1.merge_cells('I1:K1')
ws1['F1'].alignment = Alignment(horizontal='center')
ws1['I1'].alignment = Alignment(horizontal='center')

for col in ws1.columns:
    ws1.column_dimensions[get_column_letter(col[0].column)].width = max(
        (len(str(c.value)) if c.value else 0 for c in col), default=10
    ) + 2

# ── Sheet 2: Income % ─────────────────────────────────────────────────────────
ws2 = wb.create_sheet('Income_Pct_by_Scenario')

ws2.append(['income_category', 'scenario_1', 'scenario_2', 'scenario_3', 'scenario_4'])
for cell in ws2[1]: cell.font = bold

for row in inc_rows:
    ws2.append([
        row['income_category'],
        row['scenario_1_pct'], row['scenario_2_pct'],
        row['scenario_3_pct'], row['scenario_4_pct'],
    ])

for row in ws2.iter_rows(min_row=2, max_row=1 + len(inc_rows), min_col=2, max_col=5):
    for cell in row:
        cell.number_format = '0.0%'

for col in ws2.columns:
    ws2.column_dimensions[get_column_letter(col[0].column)].width = max(
        (len(str(c.value)) if c.value else 0 for c in col), default=14
    ) + 2

wb.save(out_path)
print(f"Saved: {out_path}")

## SocioEcon WFH Summary by Scenario and Year

In [ ]:
emp_cols = ['emp_retail', 'emp_srvc', 'emp_rec', 'emp_game', 'emp_other']

def summarize_socio(path):
    df = pd.read_csv(path)
    total_occ = df['total_occ_units'].sum()
    avg_pph = (
        (df['persons_per_occ_unit'] * df['total_occ_units']).sum() / total_occ
        if total_occ > 0 else 0
    )
    return {
        'total_residential_units':  int(df['total_residential_units'].sum()),
        'total_occ_units':          int(total_occ),
        'occ_units_low_inc':        int(df['occ_units_low_inc'].sum()),
        'occ_units_med_inc':        int(df['occ_units_med_inc'].sum()),
        'occ_units_high_inc':       int(df['occ_units_high_inc'].sum()),
        'avg_persons_per_occ_unit': round(avg_pph, 4),
        'total_persons':            int(df['total_persons'].sum()),
        'total_jobs':               int(df[emp_cols].sum().sum()),
    }

rows = []
for alt in [1, 2, 3, 4]:
    for year in [2035, 2050]:
        for variant, suffix in [('base', ''), ('wfh', '_wfh')]:
            path = f"{base}/Alternative_{alt}_{year}{suffix}/SocioEcon_Summer.csv"
            row = {'scenario': f"Alternative_{alt}", 'year': year, 'variant': variant}
            row.update(summarize_socio(path))
            rows.append(row)

wfh_summary = pd.DataFrame(rows)
wfh_summary.to_csv("wfh_summary.csv", index=False)

In [12]:
emp_cols = ['emp_retail', 'emp_srvc', 'emp_rec', 'emp_game', 'emp_other']

def summarize_socio(path):
    df = pd.read_csv(path)
    total_occ = df['total_occ_units'].sum()
    avg_pph = (
        (df['persons_per_occ_unit'] * df['total_occ_units']).sum() / total_occ
        if total_occ > 0 else 0
    )
    return {
        'total_residential_units':  int(df['total_residential_units'].sum()),
        'total_occ_units':          int(total_occ),
        'occ_units_low_inc':        int(df['occ_units_low_inc'].sum()),
        'occ_units_med_inc':        int(df['occ_units_med_inc'].sum()),
        'occ_units_high_inc':       int(df['occ_units_high_inc'].sum()),
        'avg_persons_per_occ_unit': round(avg_pph, 4),
        'total_persons':            int(df['total_persons'].sum()),
        'total_jobs':               int(df[emp_cols].sum().sum()),
    }

rows = []
for alt in [2, 3, 4]:
    for year in [2035, 2050]:
        for variant, suffix in [('base', ''), ('reduced', '_reduced')]:
            path = f"{base}/Alternative_{alt}{suffix}_{year}/SocioEcon_Summer.csv"
            row = {'scenario': f"Alternative_{alt}", 'year': year, 'variant': variant}
            row.update(summarize_socio(path))
            rows.append(row)

wfh_summary = pd.DataFrame(rows)
wfh_summary.to_csv("reduced_summary.csv", index=False)